In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torch_geometric "scanpy==1.11.4"

## import

In [3]:
import torch
import torch.nn.functional as F

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

Using CPU


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/GNN")
cwd = os.getcwd()
print(cwd)

sys.path.append("../../")

/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention


In [6]:
import scanpy as sc
import numpy as np
import torch
from sklearn.preprocessing import Normalizer

import matplotlib.pyplot as plt

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/local/lib/python3.12/dist-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/usr/local/lib/python3.12/dist-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/usr/local/lib/python3.12/dist-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


In [ ]:
import GNN as SA
import importlib

## Dataset

In [ ]:
adata = sc.read_h5ad("../../../Data/Breast_Cancer/ann_data.h5ad")

Noise

In [ ]:
rng = np.random.default_rng(42)
ad = adata.copy()
ad.obsm["morpho"] = rng.standard_normal((167780, 500))

In [ ]:
adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

Founsation Models

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/hoptimus_adata.h5ad")

In [ ]:
adata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow_adata.h5ad")

Preparing Dataset

In [ ]:
adata = SA.prep_adata(adata, norm=True, log1p=True)
data = SA.build_graph(adata, k=8, morph_key='p_Morpho_Embedding')

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [ ]:
print(data.x.shape)              # shape of each node
print(data.edge_index[:, :10])   # first 10 edges
print(data.edge_attr[:10])       # first 10 edge features

torch.Size([167780, 813])
tensor([[    0,     2,     0, 46964,     0,   112,     0,     4,     0,     5],
        [    2,     0, 46964,     0,   112,     0,     4,     0,     5,     0]])
tensor([ 5.7527,  5.7527,  6.9007,  6.9007,  8.1722,  8.1722,  8.4531,  8.4531,
        10.4105, 10.4105])


In [ ]:
importlib.reload(SA)
importlib.reload(SA.model)

<module 'SelfAttention.model' from '/content/drive/MyDrive/Thesis/Projects/Master_Thesis/Notebooks/SelfAttention/../../SelfAttention/model.py'>

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

## Reconstruction Loss

In [ ]:
SA.set_random_seed(42)
model = SA.SpatialTransformerAE(
    in_dim=data.x.shape[1],
    hidden_dim=64,
    latent_dim=32,
    heads=4
)

def loss_fn(pred, target, gene_dim):
    return F.mse_loss(pred[:, :gene_dim], target[:, :gene_dim])

def mask_features(data, mask_ratio=0.3):
    x = data.x.clone()
    gene_dim = data.gene_dim

    mask = torch.rand_like(x[:, :gene_dim]) < mask_ratio
    x[:, :gene_dim][mask] = 0.0

    return x

###

### h_optimus

In [ ]:
model.load_state_dict(torch.load(f'saved_models/breast_cancer_32_hoptimus_0.8.pth',weights_only=True, map_location=torch.device('cpu')))

<All keys matched successfully>

In [ ]:
## h_optimus
model = model.to(device)
data = data.to(device)

with torch.no_grad():

  x_masked = mask_features(data, 0.8)
  out = model(x_masked, data.edge_index, data.edge_attr)
  loss = loss_fn(out, data.x, data.gene_dim)
  print(loss)

tensor(0.1090)


### UNI

In [ ]:
model.load_state_dict(torch.load(f'saved_models/Breast_Cancer_32_0.8.pth',weights_only=True, map_location=torch.device('cpu')))

<All keys matched successfully>

In [ ]:
## UNI
model = model.to(device)
data = data.to(device)

with torch.no_grad():

  x_masked = mask_features(data, 0.8)
  out = model(x_masked, data.edge_index, data.edge_attr)
  loss = loss_fn(out, data.x, data.gene_dim)
  print(loss)

tensor(0.1092)
